# Automorphisms of an EPIC Stabilizer Code

This notebook shows a complete autqec workflow on a stabilizer code built with EPIC.
It demonstrates three things:
1. how to build the code in EPIC,
2. how to compute automorphisms with autqec,
3. how to convert one automorphism into a physical circuit and a logical action.

Each output object is printed with an example of how to access its fields.

In [44]:
import time
import numpy as np

from epic.modules.stabilizers_codes.css_code import CSSCode
from autqec.graph_auts import (
    B_for_all_cliffords,
    graph_aut_group,
    sparse_pcm_to_tanner_graph,
    valid_clifford_auts_B_rows,
 )
from autqec.automorphisms import circ_from_aut, logical_circ_and_pauli_correct

# Steane [[7,1,3]] CSS code from EPIC.
H = np.array(
    [[0, 0, 0, 1, 1, 1, 1],
     [0, 1, 1, 0, 0, 1, 1],
     [1, 0, 1, 0, 1, 0, 1]],
    dtype=np.uint8,
 )
lX = lZ = [1, 1, 1, 0, 0, 0, 0]

steane = CSSCode.from_css_pcm(
    code_name="steaneCode",
    hx=H,
    hz=H,
    logical_qubits=[(lX, lZ)],
)

# EPIC stores the code as a Tanner graph; autqec consumes the binary symplectic form.
H_symp, _, _ = steane.tanner_graph.parity_check_matrix
H_symp = H_symp.toarray().astype(np.uint8)
n = H_symp.shape[1] // 2

# autqec.graph_auts works on the 3-bit representation.
H_3bit = np.hstack([
    H_symp,
    (H_symp[:, :n] + H_symp[:, n:]) % 2,
 ])

print('EPIC code:', steane.name)
print('n =', steane.n, 'k =', steane.k, 'd =', steane.d)
print('H_symp shape =', H_symp.shape)
print('H_3bit shape =', H_3bit.shape)

EPIC code: steaneCode
n = 7 k = 1 d = 3
H_symp shape = (6, 14)
H_3bit shape = (6, 21)


## 1. Compute automorphisms

The graph-based autqec path computes automorphisms from the 3-bit Tanner representation.
For convenience, this notebook stores the result in a dictionary with three fields:
- `order`: the number of automorphisms found,
- `auts`: the list of generators in cycle notation,
- `time`: the runtime of the search.

In [45]:
# Compute the full automorphism-group order from the graph automorphism group.
pcm_cols = H_3bit.shape[1]
v = np.column_stack([np.arange(n), np.arange(n, 2 * n), np.arange(2 * n, 3 * n)]).flatten()
B_pcm = B_for_all_cliffords(n, bits_3=True)[:, v]
pcm_for_graph = np.vstack([H_3bit[:, v], B_pcm])
tanner_graph = sparse_pcm_to_tanner_graph(pcm_for_graph)
code_graph_auts = graph_aut_group(tanner_graph, pcm_cols, False)
group_order = code_graph_auts.order()

# Compute the Clifford-compatible automorphism generators.
t0 = time.perf_counter()
auts_epic = valid_clifford_auts_B_rows(H_3bit, bits_3=True, return_order=False)
elapsed = time.perf_counter() - t0

code_auts_dict = {
    'order': group_order,
    'auts': auts_epic,
    'time': elapsed,
}

print('code_auts_dict["order"] =', code_auts_dict['order'])
print('code_auts_dict["time"] =', code_auts_dict['time'])
print('len(code_auts_dict["auts"]) =', len(code_auts_dict['auts']))
print('code_auts_dict["auts"][0] =', code_auts_dict['auts'][0])
print('code_auts_dict["auts"][1] =', code_auts_dict['auts'][1])

code_auts_dict["order"] = 12
code_auts_dict["time"] = 0.001123958034440875
len(code_auts_dict["auts"]) = 3
code_auts_dict["auts"][0] = [[1, 10], [2, 11], [3, 12], [7, 16], [8, 17], [9, 18]]
code_auts_dict["auts"][1] = [[1, 2], [4, 5], [7, 8], [10, 11], [13, 14], [16, 17], [19, 20]]


## 2. Convert one automorphism into a physical circuit

`circ_from_aut(H_symp, aut)` turns one automorphism into a physical gate sequence on the data qubits.
`logical_circ_and_pauli_correct(H_symp, phys_circ)` then computes the induced logical action and any Pauli correction needed to keep the stabilizers consistent.

Useful fields and methods:
- `phys_act.bits_image`: the automorphism written on the 3-bit labels,
- `phys_act.ordered_qubit_triplets`: the qubit triplets after the SWAP reordering,
- `phys_act.single_qubit_gates`: the single-qubit Cliffords extracted from the automorphism,
- `phys_act.circ()`: returns `(phys_circ, symp_mat)`,
- `logical_act.U_logical_act()`: returns the logical symplectic matrix,
- `logical_act.run()`: returns `(logical_circ, phys_circ_corr)`.

In [46]:
def first_non_empty_automorphism(auts, H_symp):
    for aut in auts:
        phys_act = circ_from_aut(H_symp, aut)
        phys_circ, symp_mat = phys_act.circ()
        logical_act = logical_circ_and_pauli_correct(H_symp, phys_circ)
        logical_circ, phys_circ_corr = logical_act.run()
        if phys_circ or logical_circ:
            return aut, phys_act, phys_circ, symp_mat, logical_act, logical_circ, phys_circ_corr
    raise RuntimeError('No non-empty automorphism circuit found.')

aut0, phys_act, phys_circ, symp_mat, logical_act, logical_circ, phys_circ_corr = first_non_empty_automorphism(code_auts_dict['auts'], H_symp)

print('aut0 =', aut0)
print('type(phys_act) =', type(phys_act))
print('phys_act.bits_image =', phys_act.bits_image)
print('phys_act.ordered_qubit_triplets =', phys_act.ordered_qubit_triplets)
print('phys_act.single_qubit_gates =', phys_act.single_qubit_gates)

print('phys_circ =', phys_circ)
print('symp_mat =\n', symp_mat)

print('type(logical_act) =', type(logical_act))
print('logical_act.U_logical_act() =\n', logical_act.U_logical_act())
print('logical_circ =', logical_circ)
print('phys_circ_corr =', phys_circ_corr)

aut0 = [[1, 10], [2, 11], [3, 12], [7, 16], [8, 17], [9, 18]]
type(phys_act) = <class 'autqec.automorphisms.circ_from_aut'>
phys_act.bits_image = [10 11 12  4  5  6 16 17 18  1  2  3 13 14 15  7  8  9 19 20 21]
phys_act.ordered_qubit_triplets = [(1, 2, 3), (4, 5, 6), (7, 8, 9), (10, 11, 12), (13, 14, 15), (16, 17, 18), (19, 20, 21)]
phys_act.single_qubit_gates = []
phys_circ = [('SWAP', (3, 6)), ('SWAP', (1, 4))]
symp_mat =
 [[0 0 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1]]
type(logical_act) = <class 'autqec.automorphisms.logical_circ_and_pauli_correct'>
logical_act.U_logical_act() =
 [[1 0]
 [0 1]]
logical_

## 3. Same workflow on a more complex EPIC code (rotated surface code, d=5)

This section repeats the same autqec workflow on a larger code built with EPIC.
Compared to Steane (`n=7`), this code has `n=25` data qubits, so the
symplectic and 3-bit matrices are larger and the automorphism search is richer.

In [47]:
from epic.modules.stabilizers_codes.rotated_surface_code import RotatedSurfaceCode

# Build a larger EPIC code.
rsc5 = RotatedSurfaceCode.from_distance(distance=5, code_name='rsc5')
H_symp_rsc, _, _ = rsc5.tanner_graph.parity_check_matrix
H_symp_rsc = H_symp_rsc.toarray().astype(np.uint8)
n_rsc = H_symp_rsc.shape[1] // 2
H_3bit_rsc = np.hstack([
    H_symp_rsc,
    (H_symp_rsc[:, :n_rsc] + H_symp_rsc[:, n_rsc:]) % 2,
 ])

print('RSC code:', rsc5.name)
print('n =', rsc5.n, 'k =', rsc5.k, 'd =', rsc5.d)
print('H_symp_rsc shape =', H_symp_rsc.shape)
print('H_3bit_rsc shape =', H_3bit_rsc.shape)

# Same autqec output bundle: order / auts / time.
pcm_cols_rsc = H_3bit_rsc.shape[1]
v_rsc = np.column_stack([np.arange(n_rsc), np.arange(n_rsc, 2 * n_rsc), np.arange(2 * n_rsc, 3 * n_rsc)]).flatten()
B_pcm_rsc = B_for_all_cliffords(n_rsc, bits_3=True)[:, v_rsc]
pcm_for_graph_rsc = np.vstack([H_3bit_rsc[:, v_rsc], B_pcm_rsc])
tanner_graph_rsc = sparse_pcm_to_tanner_graph(pcm_for_graph_rsc)
code_graph_auts_rsc = graph_aut_group(tanner_graph_rsc, pcm_cols_rsc, False)
group_order_rsc = code_graph_auts_rsc.order()

t0 = time.perf_counter()
auts_rsc = valid_clifford_auts_B_rows(H_3bit_rsc, bits_3=True, return_order=False)
elapsed_rsc = time.perf_counter() - t0

code_auts_dict_rsc = {
    'order': group_order_rsc,
    'auts': auts_rsc,
    'time': elapsed_rsc,
}

print('code_auts_dict_rsc["order"] =', code_auts_dict_rsc['order'])
print('code_auts_dict_rsc["time"] =', code_auts_dict_rsc['time'])
print('len(code_auts_dict_rsc["auts"]) =', len(code_auts_dict_rsc['auts']))
print('code_auts_dict_rsc["auts"][0] =', code_auts_dict_rsc['auts'][0])

# Convert one non-trivial automorphism to physical+logical action.
aut_rsc, phys_act_rsc, phys_circ_rsc, symp_mat_rsc, logical_act_rsc, logical_circ_rsc, phys_circ_corr_rsc = first_non_empty_automorphism(
    code_auts_dict_rsc['auts'], H_symp_rsc
)

print('aut_rsc =', aut_rsc)
print('type(phys_act_rsc) =', type(phys_act_rsc))
print('phys_act_rsc.single_qubit_gates =', phys_act_rsc.single_qubit_gates)
print('phys_circ_rsc =', phys_circ_rsc)
print('symp_mat_rsc shape =', symp_mat_rsc.shape)
print('logical_act_rsc.U_logical_act() shape =', logical_act_rsc.U_logical_act().shape)
print('logical_circ_rsc =', logical_circ_rsc)
print('phys_circ_corr_rsc =', phys_circ_corr_rsc)

RSC code: rsc5
n = 25 k = 1 d = 5
H_symp_rsc shape = (24, 50)
H_3bit_rsc shape = (24, 75)
code_auts_dict_rsc["order"] = 4
code_auts_dict_rsc["time"] = 0.0019367090426385403
len(code_auts_dict_rsc["auts"]) = 1
code_auts_dict_rsc["auts"][0] = [[1, 53, 31, 38], [2, 52, 32, 37], [3, 54, 33, 39], [4, 17, 61, 59], [5, 16, 62, 58], [6, 18, 63, 60], [7, 71, 67, 50], [8, 70, 68, 49], [9, 72, 69, 51], [10, 35, 46, 26], [11, 34, 47, 25], [12, 36, 48, 27], [13, 20, 22, 29], [14, 19, 23, 28], [15, 21, 24, 30], [40, 41], [43, 56, 64, 74], [44, 55, 65, 73], [45, 57, 66, 75]]
aut_rsc = [[1, 53, 31, 38], [2, 52, 32, 37], [3, 54, 33, 39], [4, 17, 61, 59], [5, 16, 62, 58], [6, 18, 63, 60], [7, 71, 67, 50], [8, 70, 68, 49], [9, 72, 69, 51], [10, 35, 46, 26], [11, 34, 47, 25], [12, 36, 48, 27], [13, 20, 22, 29], [14, 19, 23, 28], [15, 21, 24, 30], [40, 41], [43, 56, 64, 74], [44, 55, 65, 73], [45, 57, 66, 75]]
type(phys_act_rsc) = <class 'autqec.automorphisms.circ_from_aut'>
phys_act_rsc.single_qubit_gates

In [48]:
def _normalize_targets(raw_targets):
    if isinstance(raw_targets, np.ndarray):
        raw_targets = raw_targets.tolist()

    if isinstance(raw_targets, (list, tuple)):
        values = []
        for t in raw_targets:
            if isinstance(t, np.ndarray):
                t = t.tolist()
            if isinstance(t, (list, tuple)):
                values.extend(int(x) for x in t)
            else:
                values.append(int(t))
        return tuple(values)

    return (int(raw_targets),)


def physical_circuit_to_gate_layers(physical_circuit):
    """Group an autqec physical circuit into gate-type layers.

    Returns a list of dictionaries with keys:
    - gate: gate name
    - targets: list of target-qubit tuples
    """
    layers_by_gate = {}

    for op in physical_circuit:
        if not isinstance(op, (tuple, list)) or len(op) < 2:
            raise ValueError(f"Unsupported circuit operation format: {op}")

        gate = str(op[0])
        raw_targets = op[1] if len(op) == 2 else op[1:]
        targets = _normalize_targets(raw_targets)

        if gate not in layers_by_gate:
            layers_by_gate[gate] = []
        layers_by_gate[gate].append(targets)

    return [
        {"gate": gate, "targets": targets}
        for gate, targets in layers_by_gate.items()
    ]


def print_gate_layers(physical_circuit, label="physical_circuit"):
    layers = physical_circuit_to_gate_layers(physical_circuit)
    print(f"{label}: {len(layers)} gate-type layers")
    for i, layer in enumerate(layers, start=1):
        print(f"Layer {i}: {layer['gate']} on {layer['targets']}")
    return layers


# Use whichever circuit you want to inspect.
steane_layers = print_gate_layers(phys_circ, label="phys_circ (Steane)")
rsc_layers = print_gate_layers(phys_circ_rsc, label="phys_circ_rsc (RSC d=5)")

# Programmatic access example:
# steane_layers[0] -> {'gate': 'H', 'targets': [(...)]}
# rsc_layers[1]['targets'] -> list of qubit tuples for that gate type

phys_circ (Steane): 1 gate-type layers
Layer 1: SWAP on [(3, 6), (1, 4)]
phys_circ_rsc (RSC d=5): 2 gate-type layers
Layer 1: H on [(1,), (2,), (3,), (4,), (5,), (6,), (7,), (8,), (9,), (10,), (11,), (12,), (13,), (14,), (15,), (16,), (17,), (18,), (19,), (20,), (21,), (22,), (23,), (24,), (25,)]
Layer 2: SWAP on [(23, 24), (22, 25), (20, 21), (19, 22), (17, 24), (15, 19), (13, 18), (12, 16), (11, 13), (9, 12), (8, 10), (7, 8), (6, 21), (5, 7), (4, 12), (3, 24), (2, 6), (1, 18)]
